# Chapter 14 — Node-Based Data Structures

**Source:** Wengrow, *A Common-Sense Guide to Data Structures and Algorithms*, Ch. 14

## The big idea
Arrays store items in **one contiguous block** of memory, so any index is reachable in one step.
A **linked list** stores items as separate **nodes** that can sit anywhere in memory. Each node holds:
1. its **data**
2. a **link** (pointer) to the **next node** (the last node's link is `None`)

Many later structures (trees, graphs) are built from nodes, so this chapter is the foundation.

| | Array | Linked list |
|---|---|---|
| Memory | one contiguous block | scattered nodes joined by links |
| What the computer knows at the start | the address of the first cell and the index maths | **only the first node** (the "head") | 

.
> **The most important fact in the chapter:** with a linked list you only have immediate access to the **first node**. To reach any other node you must **walk the links from the start**.

In [ ]:
class Node:
    def __init__(self, data):
        self.data = data
        self.next_node = None          # the link; None = end of the list


class LinkedList:
    def __init__(self, first_node=None):
        self.first_node = first_node   # the only thing the list itself remembers


# Build "once -> upon -> a -> time" by hand
node_1 = Node("once")
node_2 = Node("upon")
node_3 = Node("a")
node_4 = Node("time")

node_1.next_node = node_2
node_2.next_node = node_3
node_3.next_node = node_4

lst = LinkedList(node_1)

## 1. Reading and searching

**Reading index i:** start at the first node and follow `next_node` i times → **O(N)** worst case (arrays: O(1)).
**Searching for a value:** start at the first node and follow links, checking each value → **O(N)** (same as an array's linear search).

Both are loops that follow links with a `current_node` variable.
The loop ends when `current_node` is `None` (we fell off the end of the list).

In [ ]:
def read(self, index):
    current = self.first_node
    i = 0
    while i < index:
        current = current.next_node
        i += 1
        if current is None:            # index is past the end
            return None
    return current.data if current else None

def index_of(self, value):
    current = self.first_node
    i = 0
    while current:                     # stops when we fall off the end
        if current.data == value:
            return i
        current = current.next_node
        i += 1
    return None

# attach them to the class (so each cell stays small)
LinkedList.read = read
LinkedList.index_of = index_of

print(lst.read(2))            # 'a'
print(lst.read(10))           # None
print(lst.index_of("time"))   # 3
print(lst.index_of("nope"))   # None

## 2. Insertion

- **Insert at the beginning:** make a new node, point it at the old first node, make it the new `first_node`. **One step → O(1)**. (An array has to shift everything → O(N).)
- **Insert elsewhere:** the *insertion itself* is 2 pointer changes (O(1)), but you first have to **walk to the node before the spot** → **O(N)**.

Order of the two pointer changes matters:
1. `new_node.next_node = current.next_node` (new node grabs the rest of the list)
2. `current.next_node = new_node` (then the previous node links to the new node)

If you swap these, you lose the rest of the list.

In [ ]:
def insert_at_index(self, index, value):
    new_node = Node(value)

    if index == 0:                              # special case: the beginning
        new_node.next_node = self.first_node
        self.first_node = new_node
        return

    current = self.first_node
    i = 0
    while i < index - 1:                        # stop on the node BEFORE the spot
        current = current.next_node
        i += 1

    new_node.next_node = current.next_node      # 1) new node takes over the rest
    current.next_node = new_node                # 2) previous node points to the new node

LinkedList.insert_at_index = insert_at_index

def to_list(self):                              # helper to see the list as a Python list
    out, c = [], self.first_node
    while c:
        out.append(c.data)
        c = c.next_node
    return out
LinkedList.to_list = to_list

lst.insert_at_index(2, "purple")
print(lst.to_list())     # ['once', 'upon', 'purple', 'a', 'time']
lst.insert_at_index(0, "yellow")
print(lst.to_list())     # ['yellow', 'once', 'upon', 'purple', 'a', 'time']

## 3. Deletion

- **Delete the first node:** `first_node = first_node.next_node` → **O(1)**. (Array: shift everything left → O(N).)
- **Delete elsewhere:** walk to the node **before** the one you're deleting, then make its link skip over it: `current.next_node = current.next_node.next_node` → **O(N)** to get there, O(1) for the deletion itself.

The deleted node isn't erased from memory. Nothing links to it any more, and Python's garbage collector frees it.

### Arrays vs linked lists: best and worst cases are mirrored

| Scenario | Array | Linked list |
|---|---|---|
| Insert/delete at **beginning** | worst case O(N) | **best case O(1)** |
| Insert/delete in the middle | average | average |
| Insert/delete at **end** | **best case O(1)** | worst case O(N) |

| Operation | Array | Linked list |
|---|---|---|
| Read | **O(1)** | O(N) |
| Search | O(N) | O(N) |
| Insert | O(N) (O(1) at end) | O(N) (**O(1) at beginning**) |
| Delete | O(N) (O(1) at end) | O(N) (**O(1) at beginning**) |

In [ ]:
def delete_at_index(self, index):
    if index == 0:
        self.first_node = self.first_node.next_node     # drop the head
        return

    current = self.first_node
    i = 0
    while i < index - 1:                                # node BEFORE the one to delete
        current = current.next_node
        i += 1

    node_after_deleted = current.next_node.next_node    # the node after the deleted one
    current.next_node = node_after_deleted              # skip over the deleted node

LinkedList.delete_at_index = delete_at_index

lst.delete_at_index(3)
print(lst.to_list())     # ['yellow', 'once', 'upon', 'a', 'time']
lst.delete_at_index(0)
print(lst.to_list())     # ['once', 'upon', 'a', 'time']

## 4. When is a linked list actually better? "Linked lists in action"

Reaching a node costs O(N), but once you're *standing on* it, inserting or deleting is O(1). That pays off when you **pass over the whole list anyway** and change it as you go.

**Example:** remove all invalid emails from a list of 1,000 (about 100 are invalid).
- **Array:** 1,000 reads, plus each of the ~100 deletions shifts up to 1,000 elements → up to ~100,000 extra steps.
- **Linked list:** 1,000 reads plus 100 one-step deletions → **~1,100 steps**.

Rule of thumb: *one pass + many inserts/deletes along the way = linked list.*
Compare to your pandas work: filtering with a boolean mask builds a new array, which is the same trade-off.

## 5. Doubly linked lists

Each node has **two** links: `next_node` and `previous_node`.
The list keeps track of **both** `first_node` and `last_node`.

- Insert/delete/read at **either end** is **O(1)**.
- You can walk **forwards and backwards**.

Insert at the end (4 steps): create the node → its `previous_node` = old last → old last's `next_node` = new node → `last_node` = new node.
(Handle the empty list separately: then the new node is both first and last.)

## 6. A queue built on a doubly linked list

A queue (FIFO) needs **insert at end** and **remove from front**.
- Array: insert at end O(1), but remove from front **O(N)** (shifting).
- Doubly linked list: **both O(1)**, so it's the ideal underlying structure.

In Python you've already used this idea: `collections.deque` gives O(1) at both ends (Chapter 9), and it is implemented as a doubly linked structure of blocks.

In [ ]:
class DNode:
    def __init__(self, data):
        self.data = data
        self.next_node = None
        self.previous_node = None


class DoublyLinkedList:
    def __init__(self):
        self.first_node = None
        self.last_node = None

    def insert_at_end(self, value):
        new_node = DNode(value)
        if self.first_node is None:               # empty list
            self.first_node = new_node
            self.last_node = new_node
        else:
            new_node.previous_node = self.last_node
            self.last_node.next_node = new_node
            self.last_node = new_node

    def remove_from_front(self):
        removed = self.first_node
        if removed is None:
            return None
        self.first_node = removed.next_node
        if self.first_node is None:               # list became empty
            self.last_node = None
        else:
            self.first_node.previous_node = None
        return removed


class LinkedQueue:
    """FIFO queue on top of a doubly linked list. enqueue and dequeue are both O(1)."""
    def __init__(self):
        self._data = DoublyLinkedList()

    def enqueue(self, element):
        self._data.insert_at_end(element)

    def dequeue(self):
        removed = self._data.remove_from_front()
        return removed.data if removed else None

    def read(self):
        first = self._data.first_node
        return first.data if first else None


q = LinkedQueue()
for x in "abc":
    q.enqueue(x)
print(q.read(), q.dequeue(), q.dequeue(), q.dequeue(), q.dequeue())   # a a b c None

> **Differences from the book's code (on purpose):**
> - The book's `remove_from_front` doesn't reset `last_node` when the list becomes empty and doesn't clear the new head's `previous_node`. My version does both, otherwise a dequeue-then-enqueue sequence would leave stale pointers.
> - I named the class `LinkedQueue` so it doesn't clash with Python's `queue.Queue`.

## Summary

- A **node** = data + a link. A **linked list** = chain of nodes; the list object only remembers the **first node**.
- Reading/searching: **O(N)** (must walk from the head).
- Insert/delete at the **head**: **O(1)** (arrays: O(N)). At the end: O(N) (arrays: O(1)). The cases are mirror images.
- The actual re-linking is always O(1); the cost is *getting to* the right node. So linked lists shine when you're **already passing through** the list.
- **Doubly linked list**: `previous_node` + `last_node` → O(1) at both ends, can move backwards. It's the ideal base for a **queue**.
- Next chapters build more interesting structures out of nodes (not just "next").

## Exercises (book solutions: p. 453)

Use the `Node` / `LinkedList` classes from above for exercises 1, 3, 4, 5, and `DNode` / `DoublyLinkedList` for exercise 2.

### Exercise 1: print all elements
**Input:** a `LinkedList`, e.g. `once -> upon -> a -> time`.
**Task:** add a method `print_all()` that prints each element, one per line, from first to last.

### Exercise 2: print in reverse (doubly linked)
**Input:** a `DoublyLinkedList` with `"a" <-> "b" <-> "c"` (build it with `insert_at_end`).
**Task:** add `print_reverse()` that prints `c`, `b`, `a`. Use the fact that you have `last_node` and `previous_node`.

### Exercise 3: last element
**Input:** a `LinkedList` of unknown length.
**Task:** add `last()` that returns the data of the final node. You may not use a pre-stored length.

### Exercise 4: reverse the list (the "tricky one")
**Input:** a `LinkedList` `A -> B -> C`.
**Task:** add `reverse()` that changes the **links** so the list becomes `C -> B -> A` (and `first_node` points to `C`). Don't build a new list; rewire the existing nodes. Hint: you need to remember three things as you walk: the node you're on, the one before it, and the one after it.

### Exercise 5: delete a node when you only have that node
**Input:** a variable pointing to a `Node` somewhere in the **middle** of a list. You do **not** have the `LinkedList` object, and you can't reach the previous node.
**Task:** write `delete_node(node)` that effectively removes this node from the list, leaving the rest intact. Hint: you can't change the previous node's link, so think about what you *can* change on the node you were given.

In [ ]:
# Setup for exercises 1, 3, 4, 5
def make_list(values):
    nodes = [Node(v) for v in values]
    for a, b in zip(nodes, nodes[1:]):
        a.next_node = b
    return LinkedList(nodes[0]), nodes        # also return the nodes (needed for exercise 5)

lst, nodes = make_list(["once", "upon", "a", "time"])

# Exercise 1
def print_all(self):
    pass
LinkedList.print_all = print_all

# Exercise 3
def last(self):
    pass
LinkedList.last = last

# Exercise 4
def reverse(self):
    pass
LinkedList.reverse = reverse

# Exercise 5
def delete_node(node):
    pass

# --- tests ---
# lst.print_all()
# print(lst.last())                 # 'time'
# lst.reverse(); print(lst.to_list())   # ['time', 'a', 'upon', 'once']

# lst2, nodes2 = make_list(["a", "b", "c", "d"])
# delete_node(nodes2[1])            # only the node for "b" is given
# print(lst2.to_list())             # ['a', 'c', 'd']

In [ ]:
# Setup for exercise 2
dll = DoublyLinkedList()
for x in ["a", "b", "c"]:
    dll.insert_at_end(x)

def print_reverse(self):
    pass
DoublyLinkedList.print_reverse = print_reverse

# dll.print_reverse()     # expect c, b, a (one per line)